## 11.1 — מנתונים לניתוח: read_csv, to_numpy, NaN, סינון חריגים

זו הפעם הראשונה בקורס שהנתונים **לא מושלמים**: יומן מדידות אמיתי (`lab_measurements.csv`, המשך `launch_log.csv` משבוע 10) כולל מדידות שנכשלו (`NaN`) ולפחות נקודה חריגה אחת (טעות תזמון, קריאת מכשיר שגויה וכו'). לפני כל ניתוח סטטיסטי, צריך לנקות.

הכלים כבר מוכרים: `read_csv` (10.3), `()to_numpy` (10.7), ומסכות בוליאניות (7.8). החדש הוא איך משתמשים בהם יחד כדי **לנקות**, לא רק לסנן.

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
print(df.shape)

### שלב 1: איתור ערכים חסרים (`NaN`)

In [ ]:
print(df.isna().sum())          # כמה NaN בכל עמודה
df[df.isna().any(axis=1)]        # השורות שיש בהן לפחות NaN אחד

### באג נפוץ: `()dropna` בלי לשמור את התוצאה

`()df.dropna` **מחזיר טבלה חדשה** ללא השורות עם `NaN` — הוא **לא** משנה את `df` המקורי במקום (בדיוק כמו הבאג מסעיף 10.8: פעולות pandas כמעט תמיד מחזירות עותק, אלא אם מציינים `inplace=True` או משייכים בחזרה). קוד שקורא ל-`()df.dropna` בלי לשייך את התוצאה למשתנה "מנקה" — ולא באמת עושה כלום.

In [ ]:
df.dropna()                      # לא משנה את df!
print(df.isna().sum().sum())     # עדיין 2 - df המקורי לא השתנה

df_clean = df.dropna().reset_index(drop=True)   # הדרך הנכונה
print(df_clean.isna().sum().sum())               # 0

### שלב 2: סינון נקודה חריגה

סטיית תקן ("std") רגילה **רגישה** לנקודות חריגות — נקודה אחת קיצונית יכולה להגדיל אותה כל כך שהיא "מסתתרת" בתוך 3 סטיות התקן שלה. חלופה חסינה יותר: **חציון** ו-**MAD** (Median Absolute Deviation — חציון ההפרשים המוחלטים מהחציון), שלא מושפעים מנקודה בודדת קיצונית באותה מידה.

In [ ]:
def flag_outliers(group, col="range_measured", threshold=3.5):
    med = group[col].median()
    mad = (group[col] - med).abs().median()
    modified_z = 0.6745 * (group[col] - med) / mad   # "modified z-score", כלי חסין נפוץ
    return modified_z.abs() > threshold

is_outlier = df_clean.groupby("angle_deg", group_keys=False).apply(flag_outliers)
print(df_clean[is_outlier])   # הנקודה החריגה שנמצאה

df_final = df_clean[~is_outlier].reset_index(drop=True)
print(df.shape, "->", df_clean.shape, "->", df_final.shape)

### שלב 3: מעבר ל-NumPy לניתוח נומרי

In [ ]:
range_all = df_final["range_measured"].to_numpy()
print(range_all.shape, range_all.dtype)

### נסו בעצמכם

נחשו: אחרי `()dropna` ואיתור החריגה, כמה שורות אמורות להישאר ב-`df_final` (מתוך 40 המקוריות)? נמקו לפני שתריצו.

In [ ]:
# נחשו, ואז בדקו:
print(len(df_final))

`````{admonition} פתרון
:class: dropdown, tip
40 שורות מקוריות, פחות 2 עם `NaN` (`dropna`), פחות נקודה חריגה אחת (`flag_outliers`) = **37** שורות ב-`df_final`.
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה משתמשים בחציון וב-MAD, ולא בממוצע וסטיית תקן, כדי לזהות נקודות חריגות?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הם מהירים יותר לחישוב", "correct": False, "feedback": "לא זו הסיבה."},
            {"answer": "כי ממוצע וסטיית תקן עצמם מושפעים חזק מנקודה חריגה, מה שעלול 'להסתיר' אותה", "correct": True, "feedback": "נכון."},
            {"answer": "כי pandas לא תומך בחישוב סטיית תקן על עמודה", "correct": False, "feedback": "לא נכון — std נתמך היטב."}
        ]
    },
    {
        "question": "מה קורה אם קוראים ל-df.dropna() בלי לשייך את התוצאה למשתנה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "df המקורי מנוקה במקום", "correct": False, "feedback": "לא — dropna לא משנה במקום כברירת מחדל."},
            {"answer": "נוצרת טבלה חדשה ונקייה, אבל היא נזרקת מיד; df המקורי נשאר עם ה-NaN", "correct": True, "feedback": "נכון."},
            {"answer": "מתקבלת שגיאה", "correct": False, "feedback": "לא — זה קוד תקין, פשוט לא שימושי כשכותבים אותו כך."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

חשבו מחדש את `df_final`, והפעם הדפיסו גם כמה שורות הוסרו בכל שלב בנפרד (כמה עם `NaN`, כמה חריגות).

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
n_start = len(df)
df_clean = df.dropna().reset_index(drop=True)
n_after_dropna = len(df_clean)

is_outlier = df_clean.groupby("angle_deg", group_keys=False).apply(flag_outliers)
df_final = df_clean[~is_outlier].reset_index(drop=True)
n_after_outliers = len(df_final)

print(f"התחלה: {n_start}")
print(f"הוסרו עקב NaN: {n_start - n_after_dropna}")
print(f"הוסרו כחריגות: {n_after_dropna - n_after_outliers}")
print(f"נשארו: {n_after_outliers}")
```
`````